# FinGPT — Resume Backtest (PMI correction + Price fix)

## Do I have to re-run Agent 2?

**No.** The `backtest_resume_*.csv` file already contains the raw A/B/C log-probabilities
in the `signal_logits` column. We can apply PMI prior-correction directly to those numbers
— no GPU, no model, no vLLM needed.

## What was wrong and what this notebook fixes

| # | Bug | Root cause | Fix |
|---|-----|-----------|-----|
| 1 | Agent 2 always HOLD | Token "B" has LM prior ≈ 0.75 at the decision position regardless of news | **PMI correction**: subtract null-context logprobs (true model prior from disk, or empirical mean fallback) so only the news-driven component survives |
| 2 | All price fetches fail | `interval="1wk"` → 1 bar in a 7-day window → `len < 2` guard → None | Inline fetch with `interval="1d"` (5 trading days per window) |
| 3 | Stale None cache | `_RETURN_CACHE` is module-level; previous run cached None for every key | Clear cache before fetching |

## Two tracks

```
TRACK A  ── Fast (NO model required) ─────────────────────────────────
   Load backtest_resume_*.csv (already has A/B/C logits)
       │
       ▼
   PMI correction:
     • True model prior from output/pmi_null_logprobs.json  [preferred]
     • Empirical mean across CSV rows                        [fallback]
       │
       ▼
   Re-compute signal direction + probabilities
       │
       ▼
   Price fetch  (daily interval, inline, cache-cleared)
       │
       ▼
   Metrics + new CSV

TRACK B  ── Full re-run (GPU required, cells marked [B]) ───────────────
   Load original backtest_*.csv (Agent 1 results)
       │  reconstruct NewsFingerprint objects
       ▼
   Agent 2  vLLM  (A/B/C scoring + true null-logprob PMI)
       │
       ▼
   Price fetch + Metrics + CSV
```

**Run Track A cells in order: 1 → 2 → 3 → 4 → 5 → 6 → 7**  
Skip Track B cells (marked `[B]`) unless you need a full Agent 2 re-run.

In [ ]:
# ── Cell 1 · Install lightweight dependencies (Track A only needs yfinance) ──
import subprocess, sys

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *args])

_pip("yfinance", "pandas", "numpy")   # Track A — no vLLM/transformers required

# Uncomment below only if running Track B:
# _pip("vllm")
# _pip("transformers", "accelerate")
# _pip("python-dotenv", "pydantic")

print("Dependencies installed.")

In [ ]:
# ── Cell 2 · Paths ─────────────────────────────────────────────────────────
import os, sys

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Edit these paths to match your Drive layout ───────────────────────────
# The resume CSV produced by the previous run
RESUME_CSV   = "/content/drive/MyDrive/backtest_resume_20260501T033648Z.csv"

# Output directory for the corrected CSV
OUT_DIR      = "/content/drive/MyDrive"   # or "/content/"

# (Track B only) repo root and model path
REPO_DIR     = "/content/drive/MyDrive/FinGPT_Part2"
MODEL_PATH   = "/content/drive/MyDrive/deepseek_fingpt_outputs/merged_for_vllm"

# PMI prior cache written by agent2/reasoner.py after the first inference run.
# Track A loads it for the TRUE model prior; Track B writes/reads it via the
# FINGPT_PMI_PRIOR_PATH env var (set in Cell B2).
# Set to "" to always fall back to the empirical mean.
PMI_PRIOR_PATH = os.path.join(REPO_DIR, "output/pmi_null_logprobs.json")
# Optionally keep a Drive copy that survives Colab session resets:
# PMI_PRIOR_PATH = "/content/drive/MyDrive/pmi_null_logprobs.json"
# ─────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_DIR, exist_ok=True)
print(f"Resume CSV    : {RESUME_CSV}")
print(f"Output dir    : {OUT_DIR}")
print(f"PMI prior path: {PMI_PRIOR_PATH}")

In [ ]:
# ── Cell 3 · Load resume CSV and parse logits ─────────────────────────────
import ast
import numpy as np
import pandas as pd

raw_df = pd.read_csv(RESUME_CSV)
print(f"Loaded {len(raw_df)} rows.")
print(f"Columns: {list(raw_df.columns)}")

# Parse signal_logits (stored as Python list repr)
def _parse_list(val):
    if pd.isna(val) or not isinstance(val, str):
        return None
    try:
        parsed = ast.literal_eval(val)
        return parsed if isinstance(parsed, list) and len(parsed) == 3 else None
    except Exception:
        return None

raw_df["logits_list"] = raw_df["signal_logits"].apply(_parse_list)

has_logits = raw_df["logits_list"].notna()
print(f"Rows with valid A/B/C logits : {has_logits.sum()} / {len(raw_df)}")
print(f"Rows without logits (fingerprint_failed etc.): {(~has_logits).sum()}")
print()

# Original signal distribution (before PMI)
print("Original signal_direction distribution:")
print(raw_df["signal_direction"].value_counts(dropna=False).to_string())

In [ ]:
# ── Cell 4 · PMI correction (NO model needed) ─────────────────────────────
#
# Priority for the null-logprob prior:
#   1. Disk cache written by agent2/reasoner.py (TRUE model prior — best)
#   2. Empirical mean across all CSV rows (fallback — biased by news distribution)
#
# The disk cache (output/pmi_null_logprobs.json) is produced automatically on
# the first inference run.  Copy it to Drive to make it available here.

import json

CALIBRATION_T = 1.2   # softmax temperature (same as in config.py)
STRATEGY_SET  = ["BUY", "HOLD", "SELL"]
_DIR_MAP      = {0: "long", 1: "neutral", 2: "short"}
_TAG_MAP      = {0: "event_driven", 1: "none", 2: "event_driven"}

def _softmax(x, T=CALIBRATION_T):
    x = np.array(x, dtype=float) / T
    x -= x.max()
    e = np.exp(x)
    return e / e.sum()

logits_arr = np.array(
    raw_df.loc[has_logits, "logits_list"].tolist(), dtype=float
)   # shape (N, 3)

# ── 1. Try disk cache (true model prior) ─────────────────────────────────
def _load_pmi_prior(path: str):
    """Load null logprobs saved by agent2/reasoner.py.  Returns None if unavailable."""
    if not path or not os.path.exists(path):
        return None
    try:
        with open(path, encoding="utf-8") as fh:
            data = json.load(fh)
        lp = data.get("null_logprobs")
        if isinstance(lp, list) and len(lp) == 3:
            return np.array(lp, dtype=float), data
    except Exception as exc:
        print(f"  Warning: could not read PMI cache ({exc})")
    return None

_disk = _load_pmi_prior(PMI_PRIOR_PATH)

if _disk is not None:
    null_lp, _meta = _disk
    pmi_source = "disk cache (true model prior)"
    print(f"PMI prior source : {pmi_source}")
    print(f"  path           : {PMI_PRIOR_PATH}")
    print(f"  decision_prefix: {_meta.get('decision_prefix')}")
    print(f"  score_tokens   : {_meta.get('score_tokens')}")
    print(f"  computed_at    : {_meta.get('computed_at')}")
else:
    # ── 2. Empirical fallback ─────────────────────────────────────────────
    null_lp = logits_arr.mean(axis=0)
    pmi_source = "empirical mean (no disk cache found at PMI_PRIOR_PATH)"
    print(f"PMI prior source : {pmi_source}")
    print(f"  Tip: set PMI_PRIOR_PATH to output/pmi_null_logprobs.json from")
    print(f"  a previous inference run for a more accurate prior.")

print(f"\nNull logprobs [A=BUY, B=HOLD, C=SELL]: {null_lp.round(4)}")
print(f"  → B prior = {null_lp[1]:.4f} logprob  ({np.exp(null_lp[1]):.3f} raw prob)")
print()

# ── Apply PMI and re-derive signals ──────────────────────────────────────
pmi_arr = logits_arr - null_lp            # (N, 3)  — prior removed

new_rows = []
for row_idx, (df_idx, logits_pmi) in enumerate(
    zip(raw_df.index[has_logits], pmi_arr)
):
    probs  = _softmax(logits_pmi)
    best   = int(probs.argmax())
    new_rows.append({
        "_df_idx":          df_idx,
        "signal_direction": _DIR_MAP[best],
        "signal_confidence":round(float(probs[best]), 6),
        "signal_strategy_tag": _TAG_MAP[best],
        "signal_probabilities": str({
            k: round(float(v), 6)
            for k, v in zip(STRATEGY_SET, probs)
        }),
        # keep raw logits for auditing; store PMI logits too
        "pmi_logits": str(logits_pmi.tolist()),
    })

pmi_df = pd.DataFrame(new_rows).set_index("_df_idx")

# Merge PMI results back into raw_df
corrected = raw_df.copy()
corrected.loc[has_logits, "signal_direction"]     = pmi_df["signal_direction"]
corrected.loc[has_logits, "signal_confidence"]    = pmi_df["signal_confidence"]
corrected.loc[has_logits, "signal_strategy_tag"]  = pmi_df["signal_strategy_tag"]
corrected.loc[has_logits, "signal_probabilities"] = pmi_df["signal_probabilities"]
corrected.loc[has_logits, "pmi_logits"]            = pmi_df["pmi_logits"]
# Rows that had no logits keep signal_direction = NaN (skipped)

print("PMI-corrected signal_direction distribution:")
print(corrected["signal_direction"].value_counts(dropna=False).to_string())

In [ ]:
# ── Cell 5 · Fetch realized returns (self-contained, no repo code needed) ─
#
# Previous failures:
#   (a) interval='1wk' → only 1 weekly bar in a 7-day window → None
#   (b) In-memory _RETURN_CACHE from the previous run shadowed every key
#
# Both are fixed here: daily bars + cache cleared + pure inline function.

import yfinance as yf

# Clear stale in-memory cache from any prior run in this kernel session.
try:
    import importlib, backtest.price_fetcher as _pf
    _pf._RETURN_CACHE.clear()
    _pf._DISK_CACHE.clear()
    _pf._DISK_CACHE_LOADED = False
    print("In-memory price cache cleared.")
except Exception as _e:
    print(f"Cache clear skipped ({_e}) — using inline fetcher only.")

# Inline daily-interval fetcher (does not depend on any repo file).
_price_cache: dict = {}

def _fetch_return(ticker: str, start_date: str, end_date: str):
    key = (ticker, start_date, end_date)
    if key in _price_cache:
        return _price_cache[key]
    try:
        hist = yf.download(
            tickers=ticker, start=start_date, end=end_date,
            interval="1d", auto_adjust=True, progress=False,
        )
        if hist is None or hist.empty:
            _price_cache[key] = None; return None
        # Flatten MultiIndex (yfinance ≥0.2.38 single-ticker returns
        # columns like ("Close", "AXP") instead of flat "Close").
        if isinstance(hist.columns, pd.MultiIndex):
            hist.columns = hist.columns.get_level_values(0)
        if "Close" not in hist.columns:
            _price_cache[key] = None; return None
        closes = hist["Close"].dropna()
        if closes.empty:
            _price_cache[key] = None; return None
        first, last = float(closes.iloc[0]), float(closes.iloc[-1])
        ret = None if first == 0.0 else (last - first) / first
        _price_cache[key] = ret; return ret
    except Exception:
        _price_cache[key] = None; return None

# Fetch for every row in corrected (skip rows that already have NaN direction)
active = corrected[corrected["signal_direction"].notna()].copy()

realized_returns = []
fetch_ok = 0
for i, (_, row) in enumerate(active.iterrows()):
    ret = _fetch_return(str(row["ticker"]), str(row["start_date"]), str(row["end_date"]))
    realized_returns.append(ret)
    if ret is not None:
        fetch_ok += 1
    if (i + 1) % 50 == 0:
        print(f"  price fetch {i+1}/{len(active)}  ok={fetch_ok}")

print(f"\nPrice fetch done: {fetch_ok}/{len(active)} valid returns.")

In [ ]:
# ── Cell 6 · Assemble final DataFrame and compute metrics ─────────────────
import math

_POS_MAP = {"long": 1, "short": -1, "neutral": 0}

def _direction_from_return(r, threshold=0.001):
    if r > threshold:  return "up"
    if r < -threshold: return "down"
    return "neutral"

# Write realized_return, position, strategy_return, skipped_reason
active = active.copy()
active["realized_return"] = realized_returns

active["position"] = active["signal_direction"].map(_POS_MAP).fillna(0).astype(int)
active["strategy_return"] = [
    _POS_MAP.get(row["signal_direction"], 0) * row["realized_return"]
    if row["realized_return"] is not None else None
    for _, row in active.iterrows()
]
active["skipped_reason"] = [
    "" if r is not None else "price_fetch_failed"
    for r in realized_returns
]

# Rows with no logits keep their original skipped_reason
no_signal = corrected[corrected["signal_direction"].isna()].copy()
no_signal["skipped_reason"] = no_signal["skipped_reason"].fillna("signal_failed")

final = pd.concat([active, no_signal], ignore_index=True)

# ── Metrics ───────────────────────────────────────────────────────────────
successful = final[final["skipped_reason"] == ""].copy()
total, ok, skipped = len(final), len(successful), len(final) - len(successful)

if ok > 0:
    successful["realized_direction"] = successful["realized_return"].astype(float).apply(_direction_from_return)
    successful["signal_mkt_dir"]     = successful["signal_direction"].map({"long":"up","short":"down","neutral":"neutral"})
    dir_acc = float((successful["signal_mkt_dir"] == successful["realized_direction"]).mean())

    longs  = successful[successful["signal_direction"] == "long"]
    shorts = successful[successful["signal_direction"] == "short"]
    long_acc  = float((longs["realized_direction"]  == "up").mean())   if len(longs)  else float("nan")
    short_acc = float((shorts["realized_direction"] == "down").mean()) if len(shorts) else float("nan")

    sr  = successful["strategy_return"].astype(float)
    mu  = float(sr.mean())
    std = float(sr.std(ddof=0))
    sharpe = (mu / std) * math.sqrt(52) if std > 0 else float("nan")
    total_pnl = float(sr.sum())

    # vs FinGPT baseline
    fingpt_dir = successful["fingpt_label"].map({"up":"long","down":"short","neutral":"neutral"})
    vs_fingpt  = float((successful["signal_direction"] == fingpt_dir).mean())
else:
    dir_acc = long_acc = short_acc = sharpe = total_pnl = vs_fingpt = float("nan")
    mu = std = 0.0

metrics = {
    "total_rows":         total,
    "successful_rows":    ok,
    "skip_rate":          (skipped/total) if total else 0.0,
    "direction_accuracy": dir_acc,
    "long_accuracy":      long_acc,
    "short_accuracy":     short_acc,
    "mean_strategy_return": mu,
    "std_strategy_return":  std,
    "annualized_sharpe":    sharpe,
    "total_pnl":            total_pnl,
    "vs_fingpt_accuracy":   vs_fingpt,
}

print("=" * 50)
print("BACKTEST METRICS  (PMI-corrected signals)")
print("=" * 50)
for k, v in metrics.items():
    print(f"  {k:<28}: {v:.4f}" if isinstance(v, float) else f"  {k:<28}: {v}")
print()
print("Signal direction breakdown (successful rows):")
if ok > 0:
    print(successful["signal_direction"].value_counts(dropna=False).to_string())

In [ ]:
# ── Cell 7 · Save ─────────────────────────────────────────────────────────
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
out_path  = os.path.join(OUT_DIR, f"backtest_pmi_{timestamp}.csv")

# Drop the temporary parsing column before saving
final.drop(columns=["logits_list"], errors="ignore").to_csv(out_path, index=False)
print(f"Saved {len(final)} rows → {out_path}")

---
## Track B — Full Agent 2 re-run (GPU required)

Run cells B1–B6 only if you need to re-run Agent 2 from scratch (e.g. the original
`backtest_*.csv` without A/B/C logits, or you want to use the model's own
null-logprob PMI rather than the empirical estimate).

**If Track A gave sensible results, skip everything below.**

In [ ]:
# [B1] Install full dependencies (GPU runtime required)
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *a])
_pip("vllm"); _pip("transformers", "accelerate"); _pip("python-dotenv", "pydantic")
print("Done.")

In [ ]:
# [B2] Repo path + environment
import os, sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# vLLM Colab fix: in-process engine avoids sys.stdout.fileno() crash.
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["FINGPT_MODEL_PATH"]              = MODEL_PATH
os.environ["SHARE_SINGLE_LLM_BETWEEN_AGENTS"] = "1"
os.environ["FINGPT_CALIBRATION_T"]           = "1.2"
os.environ["FINGPT_LOGITS_MAX_TOKENS"]       = "1024"
os.environ["FINGPT_YF_CACHE_PATH"]           = "/tmp/yf_cache_trackB.json"

# PMI prior path: agent2/reasoner._compute_null_logprobs() will load from here
# if it already exists, or compute + save to here on first run.
# Using the same PMI_PRIOR_PATH set in Cell 2 keeps Track A and Track B in sync.
os.environ["FINGPT_PMI_PRIOR_PATH"] = PMI_PRIOR_PATH
# To persist the prior across Colab sessions, point at Drive instead:
# os.environ["FINGPT_PMI_PRIOR_PATH"] = "/content/drive/MyDrive/pmi_null_logprobs.json"

print("Environment set.")
print(f"PMI prior path : {os.environ['FINGPT_PMI_PRIOR_PATH']}")

In [ ]:
# [B3] Load ORIGINAL Agent 1 CSV and reconstruct fingerprints
import ast
import pandas as pd
from agent1.schema import NewsFingerprint

# ← Point at your original Agent 1 CSV (not the resume CSV)
AGENT1_CSV = "/content/drive/MyDrive/backtest_20260501T015749Z.csv"

a1_df  = pd.read_csv(AGENT1_CSV)
a1_ok  = a1_df[a1_df["skipped_reason"] != "fingerprint_failed"].copy()

def _parse_probs(v):
    try: return ast.literal_eval(v)
    except: return {"POSITIVE": 1/3, "NEGATIVE": 1/3, "NEUTRAL": 1/3}

a1_ok["probs_parsed"] = a1_ok["sentiment_probabilities"].apply(_parse_probs)
_L2S = {"POSITIVE": 1.0, "NEGATIVE": -1.0, "NEUTRAL": 0.0}

fingerprints, fp_idx = [], []
for idx, row in a1_ok.iterrows():
    label = str(row.get("sentiment_label", "NEUTRAL"))
    if label not in _L2S: label = "NEUTRAL"
    try:
        fp = NewsFingerprint(
            ticker=str(row.get("fingerprint_ticker", row["ticker"])),
            source="csv", published_at=str(row["start_date"]),
            headline=str(row["article_text"])[:80],
            companies_named=[str(row["ticker"])], event_keywords=[],
            sentiment_label=label, sentiment_score=_L2S[label],
            sentiment_confidence=float(row["sentiment_confidence"]),
            sentiment_probabilities=row["probs_parsed"],
            article_text=str(row["article_text"]),
        )
        fingerprints.append(fp); fp_idx.append(idx)
    except Exception as e:
        print(f"[row {idx}] {e}")
print(f"Reconstructed {len(fingerprints)} fingerprints.")

In [ ]:
# [B4] Load vLLM engine
import os, sys
from vllm import LLM
from transformers import AutoTokenizer
from agent2.reasoner import set_shared_vllm_engine
import agent2.reasoner as _a2

_nb_out = sys.stdout
try:
    sys.stdout = os.fdopen(os.dup(1), "w", buffering=1)
except Exception:
    pass
try:
    engine = LLM(model=MODEL_PATH, trust_remote_code=True, dtype="auto",
                 gpu_memory_utilization=0.85, disable_log_stats=True, enforce_eager=True)
finally:
    sys.stdout = _nb_out

set_shared_vllm_engine(engine)
_a2._chat_tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
print("Engine ready.")

In [ ]:
# [B5] Run Agent 2 in batches (PMI null logprobs computed automatically)
from agent2.reasoner import generate_signal_batch
from typing import Optional

BATCH_SIZE = 10
signals = []
for start in range(0, len(fingerprints), BATCH_SIZE):
    batch = fingerprints[start:start + BATCH_SIZE]
    end   = start + len(batch)
    print(f"Agent 2 batch {start+1}–{end}/{len(fingerprints)} ...", end=" ", flush=True)
    try:
        sigs = generate_signal_batch(batch)
    except Exception as exc:
        print(f"FAILED: {exc}")
        sigs = [None] * len(batch)
    signals.extend(sigs)
    print(f"ok={sum(s is not None for s in sigs)}/{len(batch)}")

from collections import Counter
print("Directions:", Counter((s.direction if s else None) for s in signals))

In [ ]:
# [B6] Fetch prices + assemble + metrics + save (same inline fetcher as Track A)
import yfinance as yf, pandas as pd, math, os
from datetime import datetime, timezone

_pc2: dict = {}
def _fr2(ticker, sd, ed):
    k = (ticker, sd, ed)
    if k in _pc2: return _pc2[k]
    try:
        h = yf.download(tickers=ticker, start=sd, end=ed, interval="1d",
                        auto_adjust=True, progress=False)
        if h is None or h.empty: _pc2[k]=None; return None
        if isinstance(h.columns, pd.MultiIndex): h.columns = h.columns.get_level_values(0)
        if "Close" not in h.columns: _pc2[k]=None; return None
        c = h["Close"].dropna()
        if c.empty: _pc2[k]=None; return None
        f,l = float(c.iloc[0]), float(c.iloc[-1])
        r = None if f==0 else (l-f)/f; _pc2[k]=r; return r
    except: _pc2[k]=None; return None

rows = []
for idx, fp, sig in zip(fp_idx, fingerprints, signals):
    orig = a1_ok.loc[idx]
    ret  = _fr2(str(orig["ticker"]), str(orig["start_date"]), str(orig["end_date"])) if sig else None
    pos  = {"long":1,"short":-1,"neutral":0}.get(sig.direction,0) if sig else 0
    rows.append({
        "ticker":str(orig["ticker"]), "start_date":str(orig["start_date"]), "end_date":str(orig["end_date"]),
        "article_text":str(fp.article_text)[:120], "fingpt_label":orig["fingpt_label"],
        "sentiment_label":fp.sentiment_label, "sentiment_confidence":fp.sentiment_confidence,
        "sentiment_probabilities":str(fp.sentiment_probabilities),
        "signal_direction":sig.direction if sig else None,
        "signal_confidence":sig.confidence if sig else None,
        "signal_strategy_tag":sig.strategy_tag if sig else None,
        "signal_logits":str(sig.signal_logits) if sig else None,
        "signal_probabilities":str(sig.signal_probabilities) if sig else None,
        "realized_return":ret,
        "position":pos if (sig and ret is not None) else None,
        "strategy_return":pos*ret if (sig and ret is not None) else None,
        "skipped_reason":"" if (sig and ret is not None) else ("signal_failed" if not sig else "price_fetch_failed"),
    })

res = pd.DataFrame(rows)
ts  = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
op  = os.path.join(OUT_DIR, f"backtest_full_{ts}.csv")
res.to_csv(op, index=False)
print(f"Saved → {op}")
print(res["skipped_reason"].value_counts(dropna=False).to_string())